A notebook for accumulating global statistics of the CaRDR dataset.

Author: Roger Stabbins  
Email: r.stabbins@nhm.ac.uk  
Date: 06/06/2026   
Institute: Natural History Museum, London

# Overview

1. Setup statistics table
2. Loop over entire dataset, and evaluate statistics
3. Write to global database

In [ ]:
# auto reload
%load_ext autoreload
%autoreload 2

In [ ]:
import vrdr_resampler as vrdr
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from tqdm.auto import tqdm

In [ ]:
def cube_stats(cube_path):
    """Get cube statistics

    :param cube_path: CaRDR cube path
    :type cube_path: Path
    """
    tile_id = cube_path.stem[:5]

    quad = cube_path.parent.parent.stem


    cube_stats = {}

    # Get the cube statistics
    with rasterio.open(cube_path) as src:

        # get the band names of the cube
        band_names = src.descriptions

        # lat, lon
        lat, lon = src.crs.to_dict()['lat_ts'], src.crs.to_dict()['lon_0']

        # data cube
        data_cube = src.read(masked=True)
        
        # stats
        aves = data_cube.mean(axis=(1,2))
        stds = data_cube.std(axis=(1,2))
        meds = np.median(data_cube, axis=(1,2))

        # range
        mins = data_cube.min(axis=(1,2))
        maxs = data_cube.max(axis=(1,2))


        for i, band in enumerate(band_names):
            # percentiles
            # 0.01

            band_arr = data_cube[i,:].compressed()

            q0001, q9995, q0010, q9900, q2500, q7500 = np.quantile(band_arr, [0.0001, 0.9995, 0.010, 0.990, 0.25, 0.75])
            # 99.95
            # q9995 = np.quantile(band_arr, 0.9995)
            # 1
            # q0010 = np.quantile(band_arr, 0.01)
            # 99
            # q9900 = np.quantile(band_arr, 0.99)
            # # 25th percentile
            # q2500 = np.quantile(band_arr, 0.25)
            # # 75th percentile
            # q7500 = np.quantile(band_arr, 0.75)

            # histogram over range of 0 - 1, with 2^14 bins
            hist, _ = np.histogram(band_arr, bins=2**14, range=(0, 1))

            cube_stats[band] = [
                tile_id, quad, lat, lon, 
                aves[i], stds[i], meds[i], 
                mins[i], maxs[i], 
                q0001, q9995, q0010, q9900, q2500, q7500,
                *hist]

    return cube_stats

In [ ]:
test_stats = cube_stats(cube_file)

In [ ]:
test_stats['BLU']

In [ ]:
_, bins = np.histogram([0], bins=2**14, range=(0, 1))

In [ ]:
pd.Series(test_stats['BLU'], index=col_names)

In [ ]:
# get cassis tile directory paths
vrdr_cassis_path = Path('/Volumes', 'VRDR_STORE', 'vrdr_cassis')
vrdr_path = Path('/Volumes', 'VRDR_STORE', 'vrdr') # still need to go back and get the orig. label files
mc_paths = vrdr.get_quadrant_dirs(vrdr_cassis_path)

# make a new global output directory for the stats
output_path = vrdr_cassis_path.parent / 'vrdr_global_stats'
output_path.mkdir(exist_ok=True)

for mc_path in tqdm(mc_paths, desc='Quadrants'):
    print('Processing MC Directory: ', mc_path.name)
    tile_paths = vrdr.get_tile_dirs(mc_path)

    for tile_path in tqdm(tile_paths, desc='Tiles'):
        print(' - ', tile_path.stem)
        # get tiff files
        orig_tile_path = vrdr_path / mc_path.stem / tile_path.stem
        lbl_path = Path(orig_tile_path).with_suffix('.lbl')

        out_dir = output_path / mc_path.stem
        out_dir.mkdir(parents=True, exist_ok=True)

        # read the cifcub
        cube_file = [f for f in tile_path.glob('*cifcub*.tif')][0]
        # cifcub, no_data = vrdr.read_geotiff_cube(cube_file)
        break
    break

In [ ]:
src = rasterio.open(cube_file)

In [ ]:
# get cassis tile directory paths
vrdr_cassis_path = Path('/Volumes', 'VRDR_STORE', 'vrdr_cassis')
vrdr_path = Path('/Volumes', 'VRDR_STORE', 'vrdr') # still need to go back and get the orig. label files
mc_paths = vrdr.get_quadrant_dirs(vrdr_cassis_path)

# make a new global output directory for the stats
output_path = vrdr_cassis_path.parent / 'vrdr_global_stats'
output_path.mkdir(exist_ok=True)

# make the database column names
bins = np.linspace(0,1,num=2**14, endpoint=False)
col_names = ['tile', 'quad', 'lat', 'lon', 'mean', 'std', 'median', 'min', 'max', 'q0001', 'q9995', 'q0010', 'q9900', 'q2500', 'q7500']
col_names = col_names + list(bins)

# intiate list for each band

blu_stats = []
red_stats = []
red_stats = []
nir_stats = []

for mc_path in tqdm(mc_paths, desc='Quadrants'):
    print('Processing MC Directory: ', mc_path.name)
    tile_paths = vrdr.get_tile_dirs(mc_path)

    for tile_path in tqdm(tile_paths, desc='Tiles'):
        print(' - ', tile_path.stem)
        # get tiff files
        orig_tile_path = vrdr_path / mc_path.stem / tile_path.stem
        lbl_path = Path(orig_tile_path).with_suffix('.lbl')

        # read the cifcub
        cube_file = [f for f in tile_path.glob('*cifcub*.tif')][0]
        band_stats = cube_stats(cube_file)

        # append each band to list
        blu_stats.append(band_stats['BLU'])
        red_stats.append(band_stats['red'])
        red_stats.append(band_stats['RED'])
        nir_stats.append(band_stats['NIR'])
    
    blu_df = pd.DataFrame(blu_stats, columns=col_names)
    pan_df = pd.DataFrame(pan_stats, columns=col_names)
    red_df = pd.DataFrame(red_stats, columns=col_names)
    nir_df = pd.DataFrame(nir_stats, columns=col_names)

    blu_df.to_csv(output_path / (mc_path.stem + '_blu.csv'))
    pan_df.to_csv(output_path / (mc_path.stem + '_pan.csv'))
    red_df.to_csv(output_path / (mc_path.stem + '_red.csv'))
    nir_df.to_csv(output_path / (mc_path.stem + '_nir.csv'))


Accumulating Global Statistics

Now let's loop back over this dataset, and product cumulative statistics, first for each quadrant, and then for the entire globe.

In [ ]:
# Quantile lookup — interpolate where CDF crosses the target probability
def quantile_from_hist(cdf, bin_centers, q):
    """q can be a scalar or array of probabilities in [0, 1]"""
    return np.interp(q, cdf, bin_centers)

def cumulative_stats(stats_df, quad):

    cuml = stats_df.loc[:,'0.0':].sum()
    bins = cuml.index.to_numpy().astype(np.float32)
    hist = cuml.values.astype(np.float32)

    cdf = np.cumsum(hist) / hist.sum()

    bin_centres = bins + ((bins[1] - bins[0])/2)

    median = quantile_from_hist(cdf, bin_centres, 0.5)
    q0001, q9995, q0100, q9900, q2500, q7500 = quantile_from_hist(cdf, bin_centres, [0.0001, 0.9995, 0.01, 0.99, 0.25, 0.75])

    # stats on the histogram
    min_val = bins[np.argmax(hist > 0)]
    max_val = bins[np.argmax(hist > 0) + hist[hist > 0].size - 1]

    mean = np.average(bin_centres, weights=hist)
    std  = np.sqrt(np.average((bin_centres - mean)**2, weights=hist))

    # lat is min lat across dataset
    lat_min = stats_df['lat'].min()
    lat_max = stats_df['lat'].max()

    # lon is min lon across dataset
    lon_min = stats_df['lon'].min()
    lon_max = stats_df['lon'].max()
    # quad = stats_df['quad'][0]

    cuml_stats = [
            quad, lat_min, lat_max, lon_min, lon_max, 
            mean, std, median, 
            min_val, max_val, 
            q0001, q9995, q0100, q9900, q2500, q7500,
            *hist]

    return cuml_stats

def glbl_cumulative_stats(stats_df):

    cuml = stats_df.iloc[:,16:].sum()
    bins = cuml.index.to_numpy().astype(np.float32)
    hist = cuml.values.astype(np.float32)

    cdf = np.cumsum(hist) / hist.sum()

    bin_centres = bins + ((bins[1] - bins[0])/2)

    median = quantile_from_hist(cdf, bin_centres, 0.5)
    q0001, q9995, q0100, q9900, q2500, q7500 = quantile_from_hist(cdf, bin_centres, [0.0001, 0.9995, 0.01, 0.99, 0.25, 0.75])

    # stats on the histogram
    min_val = bins[np.argmax(hist > 0)]
    max_val = bins[np.argmax(hist > 0) + hist[hist > 0].size - 1]

    mean = np.average(bin_centres, weights=hist)
    std  = np.sqrt(np.average((bin_centres - mean)**2, weights=hist))

    # lat is min lat across dataset
    lat_min = stats_df['lat_min'].min()
    lat_max = stats_df['lat_max'].max()

    # lon is min lon across dataset
    lon_min = stats_df['lon_min'].min()
    lon_max = stats_df['lon_max'].max()

    cuml_stats = [
            'global', lat_min, lat_max, lon_min, lon_max, 
            mean, std, median, 
            min_val, max_val, 
            q0001, q9995, q0100, q9900, q2500, q7500,
            *hist]

    return cuml_stats

In [ ]:
# get cassis tile directory paths
global_stats_path = Path('/Volumes', 'VRDR_STORE', 'vrdr_global_stats')

# make the database column names
bins = np.linspace(0,1,num=2**14, endpoint=False)
col_names = ['quad', 'lat_min', 'lat_max', 'lon_min', 'lon_max', 'mean', 'std', 'median', 'min', 'max', 'q0001', 'q9995', 'q0010', 'q9900', 'q2500', 'q7500']
col_names = col_names + list(bins)

# # blu data
# # get the list of all files in the global_stats_path path that end in '_blu.csv'
# blu_stats_list = list(global_stats_path.glob('mc*_blu.csv'))
# blu_cuml_stats = []
# for blu_stats in blu_stats_list:
#     blu_df = pd.read_csv(blu_stats, header=0, index_col=0)
#     blu_cuml_stats.append(cumulative_stats(blu_df, blu_stats.stem[:4]))
# blu_df = pd.DataFrame(blu_cuml_stats, columns=col_names)

# blu_df = pd.concat([blu_df, pd.DataFrame([glbl_cumulative_stats(blu_df)], columns=blu_df.columns)], ignore_index=True)

# blu_df.to_csv(global_stats_path / 'global_blu.csv')

# pan data
# get the list of all files in the global_stats_path path that end in '_pan.csv'
pan_stats_list = list(global_stats_path.glob('mc*_pan.csv'))
pan_cuml_stats = []
for pan_stats in pan_stats_list:
    pan_df = pd.read_csv(pan_stats, header=0, index_col=0)
    pan_cuml_stats.append(cumulative_stats(pan_df, pan_stats.stem[:4]))
pan_df = pd.DataFrame(pan_cuml_stats, columns=col_names)

pan_df = pd.concat([pan_df, pd.DataFrame([glbl_cumulative_stats(pan_df)], columns=pan_df.columns)], ignore_index=True)

pan_df.to_csv(global_stats_path / 'global_pan.csv')

# red data
# get the list of all files in the global_stats_path path that end in '_red.csv'
red_stats_list = list(global_stats_path.glob('mc*_red.csv'))
red_cuml_stats = []
for red_stats in red_stats_list:
    red_df = pd.read_csv(red_stats, header=0, index_col=0)
    red_cuml_stats.append(cumulative_stats(red_df, red_stats.stem[:4]))
red_df = pd.DataFrame(red_cuml_stats, columns=col_names)

red_df = pd.concat([red_df, pd.DataFrame([glbl_cumulative_stats(red_df)], columns=red_df.columns)], ignore_index=True)

red_df.to_csv(global_stats_path / 'global_red.csv')

# nir data
# get the list of all files in the global_stats_path path that end in '_nir.csv'
nir_stats_list = list(global_stats_path.glob('mc*_nir.csv'))
nir_cuml_stats = []
for nir_stats in nir_stats_list:
    nir_df = pd.read_csv(nir_stats, header=0, index_col=0)
    nir_cuml_stats.append(cumulative_stats(nir_df, nir_stats.stem[:4]))
nir_df = pd.DataFrame(nir_cuml_stats, columns=col_names)

nir_df = pd.concat([nir_df, pd.DataFrame([glbl_cumulative_stats(nir_df)], columns=nir_df.columns)], ignore_index=True)

nir_df.to_csv(global_stats_path / 'global_nir.csv')

In [ ]:
global_blu_df = pd.read_csv(global_stats_path / 'global_blu.csv')
global_pan_df = pd.read_csv(global_stats_path / 'global_pan.csv')
global_red_df = pd.read_csv(global_stats_path / 'global_red.csv')
global_nir_df = pd.read_csv(global_stats_path / 'global_nir.csv')

In [ ]:
blu_cuml = global_blu_df.iloc[-1,17:]
blu_bins = blu_cuml.index.to_numpy().astype(np.float32)
blu_hist = blu_cuml.values.astype(np.int64)

pan_cuml = global_pan_df.iloc[-1,17:]
pan_bins = pan_cuml.index.to_numpy().astype(np.float32)
pan_hist = pan_cuml.values.astype(np.int64)

red_cuml = global_red_df.iloc[-1,17:]
red_bins = red_cuml.index.to_numpy().astype(np.float32)
red_hist = red_cuml.values.astype(np.int64)

nir_cuml = global_nir_df.iloc[-1,17:]
nir_bins = nir_cuml.index.to_numpy().astype(np.float32)
nir_hist = nir_cuml.values.astype(np.int64)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
blu_hist

In [ ]:
plt.hist(blu_bins, bins=blu_bins, weights=blu_hist, alpha=0.5, label='BLU', color='turquoise')
plt.hist(pan_bins, bins=pan_bins, weights=pan_hist, alpha=0.5, label='PAN', color='grey')
plt.hist(red_bins, bins=red_bins, weights=red_hist, alpha=0.5, label='RED', color='red')
plt.hist(nir_bins, bins=nir_bins, weights=nir_hist, alpha=0.5, label='NIR', color='magenta')